# TUGAS BESAR MULTI-AGENT SYSTEM (MAS)
## Graph Pattern Matching Implementation
**Kelompok :** 4

**Mata Kuliah :** Multi-Agent System

---
**Deskripsi Proyek :**
Notebook ini berisi implementasi sistem pencarian pola pada struktur data graf menggunakan pendekatan Multi-Agent. Sistem terdiri dari tiga agen otonom:
1. **Reasoner Agent :** Menganalisis topologi data.
2. **Planner Agent :** Menyusun strategi pencarian heuristik.
3. **Searcher Agent :** Mengeksekusi algoritma VF2.

**Tujuan :**
Mendemonstrasikan efisiensi kolaborasi agen dalam menyelesaikan masalah komputasi kompleks (Graph Isomorphism).

**Data Generation & Pre-processing**

In [ ]:
import pandas as pd
import networkx as nx
import random
import numpy as np

# ======================================================
# DATA GENERATION & PRE-PROCESSING
# Kelompok 4 - Multi-Agent System Dataset
# ======================================================

def generate_dataset(num_nodes=300):
    """
    Fungsi untuk membangkitkan data graf sintetis (Erdos-Renyi)
    dan melakukan ekstraksi fitur topologi menjadi format tabular.
    """

    # 1. Raw Data Generation
    # Menggunakan model Erdos-Renyi dengan probabilitas edge 0.1
    G = nx.erdos_renyi_graph(n=num_nodes, p=0.1, seed=42)

    dataset = []

    # 2. Data Transformation (Graph to Tabular)
    # Iterasi setiap edge untuk ekstraksi fitur (Topological Sensing)
    for u, v in G.edges():

        # A. Feature Extraction
        edge_id = f"E-{u}-{v}"
        deg_source = G.degree[u]
        deg_target = G.degree[v]
        # Menghitung Clustering Coefficient sebagai indikator kepadatan lokal
        clust_coeff = round(nx.clustering(G, u), 2)

        # B. Attribute Labeling (Randomized)
        label_source = random.choice(['A', 'B', 'C', 'D'])

        # C. Derived Features (Simulasi logika agent)
        conn_type = 'Strong' if random.random() > 0.5 else 'Weak'

        # Heuristic Rule: Prioritas tinggi jika kedua node memiliki degree > 5
        priority = 'High' if (deg_source > 5 and deg_target > 5) else 'Low'

        # D. Target Labeling (Ground Truth)
        # Kondisi pola: Label 'A' dengan koneksi 'Strong'
        is_pattern = 1 if (label_source == 'A' and conn_type == 'Strong') else 0

        dataset.append([
            edge_id, u, v,
            deg_source, deg_target,
            clust_coeff, label_source,
            conn_type, priority, is_pattern
        ])

    # 3. DataFrame Construction
    columns = [
        'Edge_ID', 'Source_Node', 'Target_Node',
        'Source_Degree', 'Target_Degree',
        'Clustering_Coeff', 'Source_Label',
        'Connection_Type', 'Search_Priority', 'Is_Pattern_Match'
    ]
    df = pd.DataFrame(dataset, columns=columns)

    # 4. Data Cleaning / Downsampling
    # Membatasi dataset max 700 baris sesuai instruksi tugas
    if len(df) > 700:
        df = df.sample(700, random_state=42)

    return df

# ======================================================
# MAIN EXECUTION
# ======================================================

if __name__ == "__main__":
    print("[INFO] Memulai proses generate dataset...")

    # Generate data
    df_result = generate_dataset(300)

    # Export ke CSV
    filename = 'synthetic_graph_data.csv'
    df_result.to_csv(filename, index=False)

    print(f"[INFO] Dataset berhasil disimpan ke '{filename}'")
    print(f"-> Dimensi Data: {df_result.shape}")
    print("-> Preview Data:")
    print(df_result.head())

[INFO] Memulai proses generate dataset...
[INFO] Dataset berhasil disimpan ke 'synthetic_graph_data.csv'
-> Dimensi Data: (700, 10)
-> Preview Data:
        Edge_ID  Source_Node  Target_Node  Source_Degree  Target_Degree  \
2519  E-100-268          100          268             37             30   
2655  E-109-173          109          173             29             18   
2110   E-82-157           82          157             35             35   
151      E-5-77            5           77             23             26   
4189  E-225-245          225          245             29             33   

      Clustering_Coeff Source_Label Connection_Type Search_Priority  \
2519              0.09            C            Weak            High   
2655              0.10            C            Weak            High   
2110              0.09            C          Strong            High   
151               0.11            B            Weak            High   
4189              0.09            D          

## SYSTEM ARCHITECTURE & WORKFLOW
Berikut adalah alur kerja sistem yang diimplementasikan dalam kode ini:

```text
[DATA SOURCE]      [AGENT SYSTEM]                                     [OUTPUT]
    |                    |
(CSV/Graph) ---> [ 1. REASONER ] (Analisis Bobot Node)
                         |
                         v
                 [ 2. PLANNER  ] (Penyusunan Strategi Prioritas)
                         |
                         v
                 [ 3. SEARCHER ] (Eksekusi VF2 Algorithm) ------> [ HASIL PENCARIAN ]
                                                                  - Status: Found/Not Found
                                                                  - Execution Time
                                                                  - Mapping Location

**Multi-Agent System (MAS) Implementation & Simulation**

## MODULE 2 : AGENT ARCHITECTURE DEFINITION
Bagian ini mendefinisikan logika (Brain) dari setiap agen.
* **Reasoner :** Menggunakan *Degree Centrality* untuk menilai kompleksitas.
* **Planner :** Menggunakan Heuristik *Most Constrained First* untuk optimasi.
* **Searcher :** Menggunakan *VF2 Isomorphism Algorithm* dari library NetworkX.

In [ ]:
import pandas as pd
import networkx as nx
import time
from networkx.algorithms import isomorphism

# ======================================================
# PROGRAM IMPLEMENTASI MULTI-AGENT SYSTEM
# Topik: Graph Pattern Matching dengan Algoritma VF2
# ======================================================

# --- 1. SETUP & DATA LOADING ---
print("Initializing system environment...")

try:
    # Load dataset graph utama
    df = pd.read_csv('synthetic_graph_data.csv')

    # Konstruksi graph dari dataframe
    G_Target = nx.from_pandas_edgelist(df, 'Source_Node', 'Target_Node')
    print(f"Data Source loaded. Total Edges: {len(df)}")

except FileNotFoundError:
    print("Error: File dataset tidak ditemukan.")
    raise

# Definisi Query Graph (Pola yang dicari)
# Mengambil sampel subgraph dari node awal untuk keperluan testing
root = list(G_Target.nodes())[0]
neighbors = list(G_Target.neighbors(root))[:3]
query_nodes = [root] + neighbors
Q_Query = G_Target.subgraph(query_nodes).copy()

print(f"Target Graph Size : {G_Target.number_of_nodes()} nodes")
print(f"Query Graph Size  : {Q_Query.number_of_nodes()} nodes")
print("-" * 60)


# --- 2. CLASS DEFINITION (AGENT ARCHITECTURE) ---

class ReasonerAgent:
    """
    Agent bertanggung jawab untuk analisis metrik topologi graph.
    """
    def __init__(self):
        self.name = "Reasoner"

    def analyze_topology(self, graph):
        print(f"[{self.name}] Calculating degree centrality...")
        # Hitung derajat setiap node untuk pembobotan
        degree_stats = dict(graph.degree())
        return degree_stats

class PlannerAgent:
    """
    Agent bertanggung jawab menyusun urutan eksekusi (Heuristic Planning).
    """
    def __init__(self):
        self.name = "Planner"

    def create_search_plan(self, topology_data):
        print(f"[{self.name}] Generating execution plan...")

        # Implementasi Heuristik: Most Constrained First
        # Node dengan derajat tertinggi diproses lebih awal
        sorted_plan = sorted(topology_data, key=topology_data.get, reverse=True)

        print(f"[{self.name}] Heuristic applied. Priority queue set.")
        return sorted_plan

class SearcherAgent:
    """
    Agent eksekutor pencarian menggunakan algoritma isomorfisme VF2.
    """
    def __init__(self):
        self.name = "Searcher"

    def run_search(self, target, query, plan):
        print(f"[{self.name}] Executing VF2 algorithm...")
        start_time = time.time()

        # Inisialisasi GraphMatcher dari NetworkX
        GM = isomorphism.GraphMatcher(target, query)

        # Proses pencocokan subgraph
        if GM.subgraph_is_isomorphic():
            mapping = GM.mapping

            # Verifikasi hasil mapping berdasarkan plan
            result = {node: mapping[node] for node in plan if node in mapping}

            exec_time = time.time() - start_time
            return True, result, exec_time

        return False, {}, 0


# --- 3. MAIN EXECUTION ---
print("\nStarting Multi-Agent Simulation...")

# Instansiasi Objek Agent
reasoner = ReasonerAgent()
planner = PlannerAgent()
searcher = SearcherAgent()

# Phase 1: Reasoning
# Menganalisis struktur graf input
topology_data = reasoner.analyze_topology(Q_Query)

# Phase 2: Planning
# Menyusun strategi pencarian berdasarkan data topologi
search_plan = planner.create_search_plan(topology_data)

# Phase 3: Searching
# Melakukan pencarian pola pada graf target
found, result_map, duration = searcher.run_search(G_Target, Q_Query, search_plan)

# Output Final
print("-" * 60)
print("FINAL RESULT REPORT")
print("-" * 60)

if found:
    print("Pattern Matching Status : FOUND (SUCCESS)")
    print(f"Execution Time          : {duration:.5f} seconds")
    print("Node Mapping (Query -> Target):")
    print(result_map)
else:
    print("Pattern Matching Status : NOT FOUND")

print("-" * 60)

Initializing system environment...
Data Source loaded. Total Edges: 700
Target Graph Size : 296 nodes
Query Graph Size  : 4 nodes
------------------------------------------------------------

Starting Multi-Agent Simulation...
[Reasoner] Calculating degree centrality...
[Planner] Generating execution plan...
[Planner] Heuristic applied. Priority queue set.
[Searcher] Executing VF2 algorithm...
------------------------------------------------------------
FINAL RESULT REPORT
------------------------------------------------------------
Pattern Matching Status : FOUND (SUCCESS)
Execution Time          : 0.00041 seconds
Node Mapping (Query -> Target):
{100: 204}
------------------------------------------------------------


# **Simulation & Testing Scenarios**
Deskripsi: Pada Tahap ini merupakan eksekusi simulasi dengan empat skenario pengujian

1. Skenario 1 (Baseline): Menggunakan dataset utama (CSV) untuk memastikan sistem berjalan sesuai data yang disiapkan.

2. Skenario 2 (Small Scale): Menggunakan graf kecil (50 nodes) untuk verifikasi kebenaran logika pencarian.

3. Skenario 3 (Medium Load): Menggunakan graf menengah (500 nodes) untuk simulasi beban kerja normal.

4. Skenario 4 (High Load): Menggunakan graf besar (800 nodes) untuk menguji performa dan stabilitas agen pada skala data besar.

## MODULE 3 : AUTOMATED STRESS TEST
Bagian ini menjalankan simulasi otomatis pada 4 skenario untuk menguji validitas dan skalabilitas sistem :
1. **Baseline Test :** Menggunakan data input dari file CSV.
2. **Small Scale :** Random Graph (50 Nodes).
3. **Medium Scale :** Random Graph (500 Nodes).
4. **Large Scale (Stress Test) :** Random Graph (800 Nodes).

In [ ]:
import time
import networkx as nx
import pandas as pd

# ======================================================
# MODULE 3: AUTOMATED TESTING SIMULATION
# Executing 4 Scenarios for Performance Evaluation
# ======================================================

def run_simulation(scenario_name, graph_input):
    """
    Function to execute a single simulation cycle.
    Includes: Query Generation -> Agent Execution -> Result Logging.
    """
    print("\n" + "="*60)
    print(f"SCENARIO: {scenario_name.upper()}")
    print(f"Environment Size: {graph_input.number_of_nodes()} Nodes")
    print("-" * 60)

    # 1. SETUP QUERY PATTERN
    # Create a valid subgraph query from the existing graph
    if graph_input.number_of_nodes() > 0:
        root = list(graph_input.nodes())[0]
        neighbors = list(graph_input.neighbors(root))[:2]
        query_nodes = [root] + neighbors
        Q_Pattern = graph_input.subgraph(query_nodes).copy()

        print(f"[SETUP] Query Pattern Generated. Size: {len(query_nodes)} nodes (Root: {root})")
    else:
        print("[ERROR] Input Graph is empty.")
        return

    # 2. AGENT INITIALIZATION
    try:
        agent_reasoner = ReasonerAgent()
        agent_planner = PlannerAgent()
        agent_searcher = SearcherAgent()
    except NameError:
        print("[ERROR] Agent classes not defined. Run Module 2 first.")
        return

    # 3. EXECUTE PIPELINE
    print("\n[PROCESS] Running Multi-Agent System...")

    # Step A: Reasoning (Topology Analysis)
    stats = agent_reasoner.analyze_topology(Q_Pattern)

    # Step B: Planning (Heuristic Strategy)
    plan = agent_planner.create_search_plan(stats)

    # Step C: Searching (VF2 Execution)
    found, result, duration = agent_searcher.run_search(graph_input, Q_Pattern, plan)

    # 4. RESULT LOGGING
    print("\n[REPORT] Final Output")
    if found:
        print(f"Status        : SUCCESS (Pattern Found)")
        print(f"Duration      : {duration:.5f} seconds")
        print(f"Isomorphism Mapping (Query Node -> Target Node):")

        # Formatting result map output
        for q_node, t_node in result.items():
            print(f"   Node {q_node} \tmapped to -> Node {t_node}")

    else:
        print(f"Status        : FAILED (Pattern Not Found)")

    print("=" * 60)


# ======================================================
# MAIN EXECUTION: STRESS TEST SCENARIOS
# ======================================================

print("Starting Automated Testing Sequence...")

# --- SCENARIO 1: BASELINE TEST (FROM CSV) ---
try:
    df = pd.read_csv('synthetic_graph_data.csv')
    G_Scen1 = nx.from_pandas_edgelist(df, 'Source_Node', 'Target_Node')
    run_simulation("1. Baseline Test (Input CSV)", G_Scen1)
except FileNotFoundError:
    print("[WARNING] Scenario 1 Skipped: CSV file not found.")

# --- SCENARIO 2: SMALL SCALE TEST ---
# Generating random graph (Erdos-Renyi model)
G_Scen2 = nx.erdos_renyi_graph(n=50, p=0.15, seed=10)
run_simulation("2. Small Scale Test (50 Nodes)", G_Scen2)

# --- SCENARIO 3: MEDIUM SCALE TEST ---
G_Scen3 = nx.erdos_renyi_graph(n=500, p=0.1, seed=20)
run_simulation("3. Medium Scale Test (500 Nodes)", G_Scen3)

# --- SCENARIO 4: LARGE SCALE / STRESS TEST ---
G_Scen4 = nx.erdos_renyi_graph(n=800, p=0.08, seed=30)
run_simulation("4. Stress Test (800 Nodes)", G_Scen4)

print("\nAll simulations completed successfully.")

Starting Automated Testing Sequence...

SCENARIO: 1. BASELINE TEST (INPUT CSV)
Environment Size: 296 Nodes
------------------------------------------------------------
[SETUP] Query Pattern Generated. Size: 3 nodes (Root: 100)

[PROCESS] Running Multi-Agent System...
[Reasoner] Calculating degree centrality...
[Planner] Generating execution plan...
[Planner] Heuristic applied. Priority queue set.
[Searcher] Executing VF2 algorithm...

[REPORT] Final Output
Status        : SUCCESS (Pattern Found)
Duration      : 0.00014 seconds
Isomorphism Mapping (Query Node -> Target Node):
   Node 100 	mapped to -> Node 100

SCENARIO: 2. SMALL SCALE TEST (50 NODES)
Environment Size: 50 Nodes
------------------------------------------------------------
[SETUP] Query Pattern Generated. Size: 3 nodes (Root: 0)

[PROCESS] Running Multi-Agent System...
[Reasoner] Calculating degree centrality...
[Planner] Generating execution plan...
[Planner] Heuristic applied. Priority queue set.
[Searcher] Executing VF